# Buổi 4: Thu thập dữ liệu
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Đọc dữ liệu từ Excel nhiều sheet và JSON
- Truy vấn cơ sở dữ liệu SQL từ Python
- Gọi API web bằng thư viện requests
- Thu thập dữ liệu từ trang web bằng BeautifulSoup và pandas.read_html
- Gộp dữ liệu từ nhiều nguồn vào một DataFrame

**Dữ liệu dùng trong buổi:** `san_pham.xlsx`, `cua_hang.db`, `ty_gia_api.json`, `cua_hang_mau.html` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [1]:
import json, sqlite3
from io import StringIO
import pandas as pd
import requests
from bs4 import BeautifulSoup
DATA = "../data/"

> **Lưu ý pháp lý và đạo đức:** chỉ thu thập dữ liệu công khai, đọc điều khoản sử dụng và tệp `robots.txt` của website,
> không gửi yêu cầu dồn dập (nên `time.sleep()` giữa các lần), không thu thập dữ liệu cá nhân trái phép
> (Nghị định 13/2023/NĐ-CP về bảo vệ dữ liệu cá nhân).

## Phần A. Bài tập có hướng dẫn

### A1. Excel nhiều sheet
`sheet_name=None` đọc tất cả sheet thành một dict `{tên sheet: DataFrame}`.

In [2]:
sheets = pd.read_excel(DATA + "san_pham.xlsx", sheet_name=None)
print("Các sheet:", list(sheets))
sp = sheets["SanPham"]
ncc = sheets["NhaCungCap"]
sp.head()

Các sheet: ['SanPham', 'NhaCungCap']


,ma_sp,ten_sp,nhom_hang,gia_ban,ton_kho,ma_ncc
0,SP001,Sản phẩm 1,Sức khỏe & Làm đẹp,872000,27,NCC3
1,SP002,Sản phẩm 2,Sức khỏe & Làm đẹp,587000,231,NCC2
2,SP003,Sản phẩm 3,Điện tử,647000,125,NCC3
3,SP004,Sản phẩm 4,Gia dụng,306000,114,NCC2
4,SP005,Sản phẩm 5,Thể thao & Du lịch,408000,58,NCC2


In [3]:
# Ghép sản phẩm với nhà cung cấp theo khóa chung ma_ncc
sp_ncc = sp.merge(ncc, on="ma_ncc", how="left")
sp_ncc[["ma_sp", "ten_sp", "gia_ban", "ten_ncc", "thoi_gian_giao_ngay"]].head()

,ma_sp,ten_sp,gia_ban,ten_ncc,thoi_gian_giao_ngay
0,SP001,Sản phẩm 1,872000,Công ty Đại Việt,5
1,SP002,Sản phẩm 2,587000,Công ty An Khang,3
2,SP003,Sản phẩm 3,647000,Công ty Đại Việt,5
3,SP004,Sản phẩm 4,306000,Công ty An Khang,3
4,SP005,Sản phẩm 5,408000,Công ty An Khang,3


### A2. Truy vấn SQL từ Python
Doanh nghiệp thường lưu dữ liệu trong CSDL (MySQL, SQL Server, PostgreSQL...). Ở đây dùng SQLite (một tệp `.db`),
cách làm giống hệt: kết nối → viết câu SQL → `pd.read_sql()`.

In [4]:
con = sqlite3.connect(DATA + "cua_hang.db")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))
pd.read_sql("SELECT * FROM don_hang LIMIT 5", con)

         name
0  khach_hang
1    don_hang


,ma_don,ngay_dat,ma_kh,kenh,nhom_hang,san_pham,so_luong,don_gia,trang_thai
0,DH002819,2025-07-01,KH0448,App,Mỹ phẩm,Mỹ phẩm D,2,395000,Đã hủy
1,DH003115,2025-07-01,KH0498,App,Laptop,Laptop E,2,16609000,Hoàn thành
2,DH000716,2025-07-01,KH0110,App,Phụ kiện,Phụ kiện E,2,434000,Hoàn thành
3,DH001141,2025-07-01,KH0186,Website,Thời trang,Thời trang D,2,615000,Hoàn thành
4,DH000852,2025-07-01,KH0135,Website,Gia dụng,Gia dụng A,2,1343000,Hoàn thành


In [5]:
sql = """
SELECT k.vung, COUNT(*) AS so_don, SUM(d.so_luong * d.don_gia) / 1e6 AS doanh_thu_trieu
FROM don_hang d JOIN khach_hang k ON d.ma_kh = k.ma_kh
WHERE d.trang_thai = 'Hoàn thành'
GROUP BY k.vung
ORDER BY doanh_thu_trieu DESC
"""
pd.read_sql(sql, con).round(1)

,vung,so_don,doanh_thu_trieu
0,Miền Nam,662,3691.3
1,Miền Bắc,521,3040.1
2,Miền Trung,307,1939.2


In [6]:
# Truyền tham số an toàn bằng dấu ? (tránh lỗi SQL injection)
pd.read_sql("SELECT * FROM don_hang WHERE kenh = ? AND ngay_dat >= ?", con, params=("App", "2025-12-01")).head()

,ma_don,ngay_dat,ma_kh,kenh,nhom_hang,san_pham,so_luong,don_gia,trang_thai
0,DH002348,2025-12-01,KH0360,App,Điện thoại,Điện thoại A,2,8649000,Đã hủy
1,DH002376,2025-12-01,KH0362,App,Thời trang,Thời trang C,3,639000,Hoàn thành
2,DH002146,2025-12-02,KH0333,App,Mỹ phẩm,Mỹ phẩm D,1,556000,Hoàn thành
3,DH000119,2025-12-02,KH0020,App,Laptop,Laptop C,1,20053000,Đã hủy
4,DH001173,2025-12-02,KH0196,App,Mỹ phẩm,Mỹ phẩm B,2,461000,Hoàn thành


### A3. Gọi API
API trả dữ liệu dạng JSON. Quy trình: `requests.get(url, params=..., timeout=...)` → kiểm tra `status_code` → `.json()`.
Ô dưới thử gọi một API tỷ giá miễn phí; nếu không có mạng sẽ dùng tệp mẫu `ty_gia_api.json`.

In [7]:
try:
    r = requests.get("https://open.er-api.com/v6/latest/USD", timeout=5)
    r.raise_for_status()
    du_lieu = r.json()
    print("Nguồn: API trực tuyến –", du_lieu.get("time_last_update_utc"))
    print("1 USD =", du_lieu["rates"]["VND"], "VND")
except Exception as e:
    print("Không gọi được API (", type(e).__name__, ") → dùng dữ liệu mẫu")

Nguồn: API trực tuyến – Fri, 13 Nov 2026 00:02:31 +0000
1 USD = 25690.620978 VND


In [8]:
with open(DATA + "ty_gia_api.json", encoding="utf-8") as f:
    api = json.load(f)
print(api.keys())
print(api["rates"][0])

dict_keys(['source', 'base', 'rates'])
{'date': '2025-10-01', 'rates': {'USD': 25328.92, 'EUR': 27337.66, 'JPY': 169.08, 'CNY': 3515.09, 'KRW': 18.51, 'SGD': 19667.48}}


JSON lồng nhau → bảng phẳng bằng `pd.json_normalize`:

In [9]:
ty_gia = pd.json_normalize(api["rates"])
ty_gia.columns = [c.replace("rates.", "") for c in ty_gia.columns]
ty_gia["date"] = pd.to_datetime(ty_gia["date"])
ty_gia.head()

,date,USD,EUR,JPY,CNY,KRW,SGD
0,2025-10-01,25328.92,27337.66,169.08,3515.09,18.51,19667.48
1,2025-10-02,25490.56,27716.17,168.31,3534.55,18.40,19740.49
2,2025-10-03,25271.11,27465.45,167.91,3530.63,18.36,19604.51
3,2025-10-04,25419.11,27428.08,167.91,3518.37,18.43,19592.73
4,2025-10-05,25400.11,27458.10,167.81,3521.25,18.35,19465.16


### A4. Thu thập dữ liệu từ trang web (web scraping)
Bước 1: lấy mã HTML (trên web thật: `requests.get(url).text`; ở đây đọc tệp mẫu).
Bước 2: dùng BeautifulSoup tìm phần tử theo thẻ và class (xem bằng chuột phải → *Inspect* trên trình duyệt).

In [10]:
with open(DATA + "cua_hang_mau.html", encoding="utf-8") as f:
    html = f.read()
soup = BeautifulSoup(html, "html.parser")
print(soup.title.text)
the_dau = soup.select_one("div.product")
print(the_dau.prettify()[:400])

Cửa hàng Phụ kiện Mẫu
<div class="product" data-id="P001">
 <h3 class="name">
  Tai nghe Bluetooth
 </h3>
 <span class="price">
  2.935.000₫
 </span>
 <span class="old-price">
  4.372.000₫
 </span>
 <div class="rating">
  3.3
 </div>
 <div class="sold">
  Đã bán 2008
 </div>
 <div class="stock">
  Còn hàng
 </div>
</div>



In [11]:
def so(chuoi):
    """'1.250.000₫' -> 1250000"""
    return int("".join(ch for ch in chuoi if ch.isdigit()))

ds = []
for p in soup.select("div.product"):
    ds.append({
        "ma": p["data-id"],
        "ten": p.select_one(".name").text.strip(),
        "gia": so(p.select_one(".price").text),
        "gia_cu": so(p.select_one(".old-price").text),
        "danh_gia": float(p.select_one(".rating").text),
        "da_ban": so(p.select_one(".sold").text),
        "con_hang": p.select_one(".stock").text == "Còn hàng",
    })
san_pham_web = pd.DataFrame(ds)
san_pham_web["giam_%"] = ((1 - san_pham_web["gia"] / san_pham_web["gia_cu"]) * 100).round(1)
san_pham_web.head()

,ma,ten,gia,gia_cu,danh_gia,da_ban,con_hang,giam_%
0,P001,Tai nghe Bluetooth,2935000,4372000,3.3,2008,True,32.9
1,P002,Sạc dự phòng 10000mAh,3311000,3653000,3.5,3935,False,9.4
2,P003,Chuột không dây,2025000,2838000,3.3,3748,False,28.6
3,P004,Bàn phím cơ,2684000,3868000,4.0,4458,True,30.6
4,P005,Loa mini,1119000,1323000,4.5,849,True,15.4


Bảng HTML (`<table>`) đọc nhanh bằng `pd.read_html` – trả về list các DataFrame:

In [12]:
bang = pd.read_html(StringIO(html), attrs={"id": "shipping"})[0]
bang

,Khu vực,Phí (nghìn đồng),Thời gian (ngày)
0,Nội thành TP.HCM,15,1
1,Ngoại thành TP.HCM,25,2
2,Miền Nam,30,2
3,Miền Trung,35,3
4,Miền Bắc,40,4


### A5. Gộp nhiều nguồn và lưu kết quả
Quy đổi giá sản phẩm thu thập từ web sang USD theo tỷ giá trung bình tháng từ API, rồi lưu thành CSV.

In [13]:
usd_tb = ty_gia["USD"].mean()
san_pham_web["gia_usd"] = (san_pham_web["gia"] / usd_tb).round(2)
san_pham_web.to_csv("san_pham_thu_thap.csv", index=False, encoding="utf-8-sig")
print(f"Tỷ giá TB: {usd_tb:,.0f} VND/USD")
san_pham_web[["ten", "gia", "gia_usd"]].head()

Tỷ giá TB: 25,357 VND/USD


,ten,gia,gia_usd
0,Tai nghe Bluetooth,2935000,115.75
1,Sạc dự phòng 10000mAh,3311000,130.58
2,Chuột không dây,2025000,79.86
3,Bàn phím cơ,2684000,105.85
4,Loa mini,1119000,44.13


## Phần B. Bài tập tự làm

### Bài B1. Hàng sắp hết và nhà cung cấp

Từ `san_pham.xlsx`: liệt kê sản phẩm có tồn kho dưới 30, kèm tên nhà cung cấp, tỉnh thành và thời gian giao hàng.
Sắp xếp theo tồn kho tăng dần. Nhà cung cấp nào có nhiều sản phẩm sắp hết nhất?

In [14]:
# Viết code của bạn ở đây
sheets = pd.read_excel(DATA + "san_pham.xlsx", sheet_name=None)
sp  = sheets["SanPham"]
ncc = sheets["NhaCungCap"]

print("Cột SanPham   :", sp.columns.tolist())
print("Cột NhaCungCap:", ncc.columns.tolist())
cot_ton   = [c for c in sp.columns  if "ton" in c.lower()][0]
cot_tinh  = [c for c in ncc.columns if "tinh" in c.lower() or "thanh" in c.lower()][0]
print("Cột tồn kho:", cot_ton, "| Cột tỉnh thành:", cot_tinh)
sp_ncc = sp.merge(ncc, on="ma_ncc", how="left")
sap_het = sp_ncc[sp_ncc[cot_ton] < 30].sort_values(cot_ton, ascending=True)
kq = sap_het[["ma_sp", "ten_sp", cot_ton, "ten_ncc", cot_tinh, "thoi_gian_giao_ngay"]]
display(kq)
dem = kq.groupby("ten_ncc").size().sort_values(ascending=False)
print(dem)
print("Nhà cung cấp có nhiều sản phẩm sắp hết nhất:", dem.index[0], "-", dem.iloc[0], "sản phẩm")

Cột SanPham   : ['ma_sp', 'ten_sp', 'nhom_hang', 'gia_ban', 'ton_kho', 'ma_ncc']
Cột NhaCungCap: ['ma_ncc', 'ten_ncc', 'tinh_thanh', 'thoi_gian_giao_ngay']
Cột tồn kho: ton_kho | Cột tỉnh thành: tinh_thanh


,ma_sp,ten_sp,ton_kho,ten_ncc,tinh_thanh,thoi_gian_giao_ngay
16,SP017,Sản phẩm 17,3,Công ty Sao Mai,Đà Nẵng,4
39,SP040,Sản phẩm 40,15,Công ty Đại Việt,Hà Nội,5
9,SP010,Sản phẩm 10,24,Công ty Sao Mai,Đà Nẵng,4
0,SP001,Sản phẩm 1,27,Công ty Đại Việt,Hà Nội,5


ten_ncc
Công ty Sao Mai     2
Công ty Đại Việt    2
dtype: int64
Nhà cung cấp có nhiều sản phẩm sắp hết nhất: Công ty Sao Mai - 2 sản phẩm


### Bài B2. Viết truy vấn SQL

Từ `cua_hang.db`, viết câu SQL (dùng `pd.read_sql`) để lấy: doanh thu (triệu đồng) và số đơn **hoàn thành** theo **kênh bán**,
sắp xếp giảm dần. Sau đó viết truy vấn thứ hai lấy 10 khách hàng có tổng chi tiêu cao nhất (mã KH, họ tên, vùng, tổng chi tiêu).

In [15]:
# Viết code của bạn ở đây
con = sqlite3.connect(DATA + "cua_hang.db")

# Truy vấn 1: doanh thu (triệu đồng) và số đơn hoàn thành theo kênh bán
sql1 = """
SELECT kenh,
       COUNT(*) AS so_don_hoan_thanh,
       SUM(so_luong * don_gia) / 1e6 AS doanh_thu_trieu
FROM don_hang
WHERE trang_thai = 'Hoàn thành'
GROUP BY kenh
ORDER BY doanh_thu_trieu DESC
"""
display(pd.read_sql(sql1, con).round(1))

# Truy vấn 2: 10 khách hàng có tổng chi tiêu cao nhất
sql2 = """
SELECT k.ma_kh, k.ho_ten, k.vung,
       SUM(d.so_luong * d.don_gia) AS tong_chi_tieu
FROM don_hang d JOIN khach_hang k ON d.ma_kh = k.ma_kh
WHERE d.trang_thai = 'Hoàn thành'
GROUP BY k.ma_kh, k.ho_ten, k.vung
ORDER BY tong_chi_tieu DESC
LIMIT 10
"""
display(pd.read_sql(sql2, con))

,kenh,so_don_hoan_thanh,doanh_thu_trieu
0,App,449,2502.7
1,Website,399,2412.1
2,Sàn TMĐT,418,2397.3
3,Cửa hàng,224,1358.6


,ma_kh,ho_ten,vung,tong_chi_tieu
0,KH0475,Khách hàng 475,Miền Bắc,196091000
1,KH0455,Khách hàng 455,Miền Bắc,170637000
2,KH0424,Khách hàng 424,Miền Trung,151071000
3,KH0417,Khách hàng 417,Miền Bắc,143247000
4,KH0020,Khách hàng 20,Miền Trung,132441000
5,KH0264,Khách hàng 264,Miền Bắc,127181000
6,KH0331,Khách hàng 331,Miền Trung,125428000
7,KH0087,Khách hàng 87,Miền Bắc,121957000
8,KH0269,Khách hàng 269,Miền Trung,118968000
9,KH0224,Khách hàng 224,Miền Nam,117175000


### Bài B3. Phân tích tỷ giá

Từ DataFrame `ty_gia` (Phần A3): tìm ngày USD cao nhất và thấp nhất trong tháng, biên độ dao động (%),
và tính % thay đổi của mỗi đồng tiền từ đầu tháng đến cuối tháng.

In [16]:
# Viết code của bạn ở đây
tg = ty_gia.sort_values("date").reset_index(drop=True)

ngay_cao  = tg.loc[tg["USD"].idxmax()]
ngay_thap = tg.loc[tg["USD"].idxmin()]
print("USD cao nhất :", ngay_cao["USD"], "vào ngày", ngay_cao["date"].date())
print("USD thấp nhất:", ngay_thap["USD"], "vào ngày", ngay_thap["date"].date())

# 2. Biên độ dao động (%) = (cao nhất - thấp nhất) / thấp nhất * 100
bien_do = (ngay_cao["USD"] - ngay_thap["USD"]) / ngay_thap["USD"] * 100
print(f"Biên độ dao động USD: {bien_do:.2f}%")

# 3. % thay đổi của mỗi đồng tiền từ đầu tháng đến cuối tháng
cot_tien = tg.columns.drop("date")
dau, cuoi = tg[cot_tien].iloc[0], tg[cot_tien].iloc[-1]
thay_doi = ((cuoi - dau) / dau * 100).round(2)

kq = thay_doi.sort_values(ascending=False).rename("thay_doi_%").to_frame()
display(kq)

USD cao nhất : 25497.31 vào ngày 2025-10-24
USD thấp nhất: 25180.84 vào ngày 2025-10-08
Biên độ dao động USD: 1.26%


,thay_doi_%
USD,0.39
JPY,0.35
EUR,0.28
CNY,0.09
SGD,0.07
KRW,-0.97


### Bài B4. Phân tích sản phẩm thu thập từ web

Từ `san_pham_web`: (a) 5 sản phẩm bán chạy nhất; (b) sản phẩm giảm giá sâu nhất; (c) điểm đánh giá trung bình của
sản phẩm còn hàng và hết hàng; (d) ước tính doanh thu = giá × số đã bán, sản phẩm nào cao nhất?

In [17]:
# Viết code của bạn ở đây
print("(a) Top 5 bán chạy nhất:")
display(san_pham_web.nlargest(5, "da_ban")[["ma", "ten", "gia", "da_ban"]])

sp_giam = san_pham_web.loc[san_pham_web["giam_%"].idxmax()]
print("(b) Giảm giá sâu nhất:", sp_giam["ten"], "- giảm", sp_giam["giam_%"], "%")

print("(c) Đánh giá trung bình theo tình trạng hàng:")
dg = san_pham_web.groupby("con_hang")["danh_gia"].mean().round(2)
dg.index = dg.index.map({True: "Còn hàng", False: "Hết hàng"})
display(dg)

san_pham_web["doanh_thu_uoc_tinh"] = san_pham_web["gia"] * san_pham_web["da_ban"]
sp_dt = san_pham_web.loc[san_pham_web["doanh_thu_uoc_tinh"].idxmax()]
print("(d) Doanh thu ước tính cao nhất:", sp_dt["ten"], "-", f"{sp_dt['doanh_thu_uoc_tinh']:,.0f}", "đ")
display(san_pham_web.nlargest(5, "doanh_thu_uoc_tinh")[["ten", "gia", "da_ban", "doanh_thu_uoc_tinh"]])

(a) Top 5 bán chạy nhất:


,ma,ten,gia,da_ban
9,P010,Webcam HD,116000,4860
3,P004,Bàn phím cơ,2684000,4458
5,P006,Đồng hồ thông minh,3039000,4347
1,P002,Sạc dự phòng 10000mAh,3311000,3935
15,P016,Quạt mini USB,1885000,3872


(b) Giảm giá sâu nhất: Tai nghe Bluetooth - giảm 32.9 %
(c) Đánh giá trung bình theo tình trạng hàng:


con_hang
Hết hàng    4.10
Còn hàng    4.08
Name: danh_gia, dtype: float64

(d) Doanh thu ước tính cao nhất: Đồng hồ thông minh - 13,210,533,000 đ


,ten,gia,da_ban,doanh_thu_uoc_tinh
5,Đồng hồ thông minh,3039000,4347,13210533000
1,Sạc dự phòng 10000mAh,3311000,3935,13028785000
16,Hub USB-C,3467000,3579,12408393000
3,Bàn phím cơ,2684000,4458,11965272000
10,Thẻ nhớ 64GB,3275000,2486,8141650000


### Bài B5. Tính phí giao hàng

Dùng bảng phí vận chuyển `bang` (Phần A4). Viết hàm `phi_ship(khu_vuc, gia_tri_don)` trả về phí (nghìn đồng):
miễn phí nếu đơn ≥ 500 nghìn đồng, ngược lại lấy theo bảng. Thử với ("Miền Bắc", 320) và ("Nội thành TP.HCM", 650).

In [18]:
# Viết code của bạn ở đây
display(bang)
cot_kv  = bang.columns[0]
cot_phi = [c for c in bang.columns if "phí" in str(c).lower() or "phi" in str(c).lower()][0]

def lay_so(x):
    """'30.000đ' -> 30000 ; '30' -> 30"""
    s = "".join(ch for ch in str(x) if ch.isdigit())
    return int(s) if s else 0

bang_phi = {}
for _, r in bang.iterrows():
    phi = lay_so(r[cot_phi])
    bang_phi[str(r[cot_kv]).strip()] = phi / 1000 if phi >= 1000 else phi

def phi_ship(khu_vuc, gia_tri_don):
    """Trả về phí giao hàng (nghìn đồng). Đơn >= 500 nghìn: miễn phí."""
    if gia_tri_don >= 500:
        return 0
    if khu_vuc not in bang_phi:
        raise ValueError(f"Không có khu vực '{khu_vuc}' trong bảng phí: {list(bang_phi)}")
    return bang_phi[khu_vuc]

print("Miền Bắc, 320        ->", phi_ship("Miền Bắc", 320), "nghìn đồng")
print("Nội thành TP.HCM, 650 ->", phi_ship("Nội thành TP.HCM", 650), "nghìn đồng")

,Khu vực,Phí (nghìn đồng),Thời gian (ngày)
0,Nội thành TP.HCM,15,1
1,Ngoại thành TP.HCM,25,2
2,Miền Nam,30,2
3,Miền Trung,35,3
4,Miền Bắc,40,4


Miền Bắc, 320        -> 40 nghìn đồng
Nội thành TP.HCM, 650 -> 0 nghìn đồng


## Phần C. Bài tập làm thêm

### Bài C1. Hàm thu thập có xử lý lỗi

Viết hàm `lay_json(url, tep_du_phong)` thử gọi API (timeout 5 giây, tối đa 3 lần, nghỉ 1 giây giữa các lần);
nếu thất bại thì đọc tệp dự phòng. Hàm trả về dict và in ra nguồn dữ liệu đã dùng.

In [19]:
# Viết code của bạn ở đây
import time

def lay_json(url, tep_du_phong):
    """Thử gọi API tối đa 3 lần (timeout 5s, nghỉ 1s giữa các lần).
    Thất bại hết thì đọc tệp dự phòng. Trả về dict và in nguồn dữ liệu đã dùng."""
    for lan in range(1, 4):
        try:
            r = requests.get(url, timeout=5)
            r.raise_for_status()
            du_lieu = r.json()
            print(f"Nguồn dữ liệu: API trực tuyến (thành công ở lần thử {lan})")
            return du_lieu
        except Exception as e:
            print(f"Lần {lan}/3 thất bại: {type(e).__name__}")
            if lan < 3:
                time.sleep(1)

    # Cả 3 lần đều lỗi -> đọc tệp dự phòng
    with open(tep_du_phong, encoding="utf-8") as f:
        du_lieu = json.load(f)
    print(f"Nguồn dữ liệu: tệp dự phòng ({tep_du_phong})")
    return du_lieu

d1 = lay_json("https://open.er-api.com/v6/latest/USD", DATA + "ty_gia_api.json")
print(type(d1), list(d1.keys())[:5])

d2 = lay_json("https://khong-ton-tai.invalid/api", DATA + "ty_gia_api.json")
print(type(d2), list(d2.keys())[:5])

Nguồn dữ liệu: API trực tuyến (thành công ở lần thử 1)
<class 'dict'> ['result', 'provider', 'documentation', 'terms_of_use', 'time_last_update_unix']
Lần 1/3 thất bại: ConnectionError
Lần 2/3 thất bại: ConnectionError
Lần 3/3 thất bại: ConnectionError
Nguồn dữ liệu: tệp dự phòng (../data/ty_gia_api.json)
<class 'dict'> ['source', 'base', 'rates']


### Bài C2. Thu thập từ một trang web thật

Chọn một trang có bảng dữ liệu công khai (ví dụ trang Wikipedia "Danh sách đơn vị hành chính cấp tỉnh của Việt Nam").
Dùng `requests` + `pd.read_html` lấy bảng, làm sạch tên cột, và cho biết 5 tỉnh có dân số lớn nhất.
Ghi rõ URL và ngày thu thập trong notebook. (Cần kết nối Internet; nếu trang thay đổi cấu trúc, hãy tự điều chỉnh.)

### Bài C3. Lưu dữ liệu thu thập vào CSDL

Tạo CSDL mới `du_lieu_thu_thap.db`, ghi `san_pham_web` thành bảng `san_pham` và `ty_gia` thành bảng `ty_gia`
(`DataFrame.to_sql`). Sau đó viết truy vấn SQL lấy các sản phẩm còn hàng có giá dưới 500.000 đ, kèm giá USD
tính theo tỷ giá ngày cuối cùng trong bảng `ty_gia`.

In [20]:
# Viết code của bạn ở đây
